# DART JAX VAE partial TPU run

This notebook runs the JAX VAE on Kaggle TPU v5e-8. It has two trainer modes:

- `single`: known-good single-replica `nnx.jit` path
- `spmd`: data-sharded JAX mesh/GSPMD path for using the 8-device TPU more properly

Upload/mount these inputs:

- TPU shard dataset with `metadata.json`, `normalization.npz`, and `shards/`
- VAE checkpoint `latest.npz` or `checkpoint_*.npz`
- SMPL-X locked-head directory containing `smplx/SMPLX_MALE.npz`
- the DART repo snapshot containing `jax_dart/`

In [34]:
# Edit these paths for your Kaggle mount names.
REPO_IN = "/kaggle/input/models/markuskamsties/codes/jax/default/1"
REPO = "/kaggle/working/DART"

ROOT = "/kaggle/input/datasets/markuskamsties/testrunning"
INIT = "/kaggle/input/datasets/markuskamsties/testrunning/latest.npz"
SMPL = "/kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead"

OUT_SINGLE = "/kaggle/working/runs/jax_vae_tpu_single_smoke"
OUT_SPMD = "/kaggle/working/runs/jax_vae_tpu_spmd_partial"

import os
os.environ.update({
    "REPO_IN": REPO_IN,
    "REPO": REPO,
    "ROOT": ROOT,
    "INIT": INIT,
    "SMPL": SMPL,
    "OUT_SINGLE": OUT_SINGLE,
    "OUT_SPMD": OUT_SPMD,
})

print("REPO_IN:", REPO_IN)
print("ROOT:", ROOT)
print("INIT:", INIT)
print("SMPL:", SMPL)

REPO_IN: /kaggle/input/models/markuskamsties/codes/jax/default/1
ROOT: /kaggle/input/datasets/markuskamsties/testrunning
INIT: /kaggle/input/datasets/markuskamsties/testrunning/latest.npz
SMPL: /kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead


## Copy repo to writable working space

In [35]:
!rm -rf "$REPO"
!cp -r "$REPO_IN" "$REPO"
%cd /kaggle/working/DART
!ls -lah jax_dart/training
!ls -lah jax_dart

!ls -lah jax_dart/kaggle || true

sh: 0: getcwd() failed: No such file or directory
/kaggle/working/DART
total 76K
drwxr-xr-x 2 root root 4.0K Apr 28 23:20 .
drwxr-xr-x 8 root root 4.0K Apr 28 23:20 ..
-rw-r--r-- 1 root root   69 Apr 28 23:20 __init__.py
-rw-r--r-- 1 root root  17K Apr 28 23:20 vae_smoke.py
-rw-r--r-- 1 root root  21K Apr 28 23:20 vae_train.py
-rw-r--r-- 1 root root  20K Apr 28 23:20 vae_train_spmd.py
total 36K
drwxr-xr-x  8 root root 4.0K Apr 28 23:20 .
drwxr-xr-x 20 root root 4.0K Apr 28 23:20 ..
-rw-r--r--  1 root root   49 Apr 28 23:20 __init__.py
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 data
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 kaggle
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 models
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 parity
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 training
drwxr-xr-x  2 root root 4.0K Apr 28 23:20 visualization
total 20K
drwxr-xr-x 2 root root 4.0K Apr 28 23:20 .
drwxr-xr-x 8 root root 4.0K Apr 28 23:20 ..
-rw-r--r-- 1 root root   61 Apr 28 23:20 __init__.py
-rw-r--r

## TPU/path check

In [36]:
!python jax_dart/kaggle/vae_partial_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --trainer spmd \
  --check-only \
  --root "$ROOT" \
  --out-dir "$OUT_SPMD" \
  --init-npz "$INIT" \
  --smpl-model-dir "$SMPL"

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
root: /kaggle/input/datasets/markuskamsties/testrunning (ok)
metadata: /kaggle/input/datasets/markuskamsties/testrunning/metadata.json (ok)
normalization: /kaggle/input/datasets/markuskamsties/testrunning/normalization.npz (ok)
init_npz: /kaggle/input/datasets/markuskamsties/testrunning/latest.npz (ok)
smpl_model_dir: /kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead (ok)
out_dir: /kaggle/working/runs/jax_vae_tpu_spmd_partial
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777418424.658210    7427 common_lib.cc:650] Could not set metric server port: INVALID_ARGUMENT: Coul

## Known-good single-replica smoke

Run this first if you want the same fallback path that already worked on Kaggle.

In [37]:
!python jax_dart/kaggle/vae_partial_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --trainer single \
  --root "$ROOT" \
  --out-dir "$OUT_SINGLE" \
  --init-npz "$INIT" \
  --batch-samples 16 \
  --steps 100 \
  --log-every 10 \
  --eval-every 50 \
  --save-every 100 \
  --eval-batches 4 \
  --dtype bf16 \
  --h-dim 256 \
  --num-layers 7 \
  --latent-dim "1 256" \
  --learning-rate 1e-4 \
  --weight-joints-delta 1.0 \
  --weight-transl-delta 1.0 \
  --weight-orient-delta 2.0 \
  --weight-smpl-joints-rec 1.0 \
  --weight-joints-consistency 0.1 \
  --smpl-model-dir "$SMPL" \
  --fail-on-nonfinite

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
root: /kaggle/input/datasets/markuskamsties/testrunning (ok)
metadata: /kaggle/input/datasets/markuskamsties/testrunning/metadata.json (ok)
normalization: /kaggle/input/datasets/markuskamsties/testrunning/normalization.npz (ok)
init_npz: /kaggle/input/datasets/markuskamsties/testrunning/latest.npz (ok)
smpl_model_dir: /kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead (ok)
out_dir: /kaggle/working/runs/jax_vae_tpu_single_smoke
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777418445.088600    8182 common_lib.cc:650] Could not set metric server port: INVALID_ARGUMENT: Coul

## SPMD smoke

This is the first real multi-device path. `batch_samples=64` gives a global VAE batch of `64 * 4 = 256`, so on 8 devices the per-device batch is 32.

In [38]:
!python jax_dart/kaggle/vae_partial_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --trainer spmd \
  --root "$ROOT" \
  --out-dir "$OUT_SPMD" \
  --init-npz "$INIT" \
  --batch-samples 64 \
  --steps 100 \
  --log-every 10 \
  --eval-every 50 \
  --save-every 100 \
  --eval-batches 4 \
  --dtype bf16 \
  --h-dim 256 \
  --num-layers 7 \
  --latent-dim "1 256" \
  --learning-rate 1e-4 \
  --weight-joints-delta 1.0 \
  --weight-transl-delta 1.0 \
  --weight-orient-delta 2.0 \
  --weight-smpl-joints-rec 1.0 \
  --weight-joints-consistency 0.1 \
  --smpl-model-dir "$SMPL" \
  --fail-on-nonfinite

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
root: /kaggle/input/datasets/markuskamsties/testrunning (ok)
metadata: /kaggle/input/datasets/markuskamsties/testrunning/metadata.json (ok)
normalization: /kaggle/input/datasets/markuskamsties/testrunning/normalization.npz (ok)
init_npz: /kaggle/input/datasets/markuskamsties/testrunning/latest.npz (ok)
smpl_model_dir: /kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead (ok)
out_dir: /kaggle/working/runs/jax_vae_tpu_spmd_partial
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777418542.479252    9438 common_lib.cc:650] Could not set metric server port: INVALID_ARGUMENT: Coul

## SPMD partial run

In [50]:
STEPS = 5000
os.environ["STEPS"] = str(STEPS)

!python jax_dart/kaggle/vae_partial_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --trainer spmd \
  --root "$ROOT" \
  --out-dir "$OUT_SPMD" \
  --init-npz "$INIT" \
  --batch-samples 512 \
  --steps "$STEPS" \
  --log-every 50 \
  --eval-every 500 \
  --save-every 1000 \
  --eval-batches 16 \
  --dtype bf16 \
  --h-dim 256 \
  --num-layers 7 \
  --latent-dim "1 256" \
  --learning-rate 1e-4 \
  --weight-joints-delta 1.0 \
  --weight-transl-delta 1.0 \
  --weight-orient-delta 2.0 \
  --weight-smpl-joints-rec 1.0 \
  --weight-joints-consistency 0.1 \
  --smpl-model-dir "$SMPL" \
  --fail-on-nonfinite

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
root: /kaggle/input/datasets/markuskamsties/testrunning (ok)
metadata: /kaggle/input/datasets/markuskamsties/testrunning/metadata.json (ok)
normalization: /kaggle/input/datasets/markuskamsties/testrunning/normalization.npz (ok)
init_npz: /kaggle/input/datasets/markuskamsties/testrunning/latest.npz (ok)
smpl_model_dir: /kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead (ok)
out_dir: /kaggle/working/runs/jax_vae_tpu_spmd_partial
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777420089.021767   21869 common_lib.cc:650] Could not set metric server port: INVALID_ARGUMENT: Coul

## Reload eval

In [ ]:
!python jax_dart/kaggle/vae_partial_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --trainer spmd \
  --root "$ROOT" \
  --out-dir "${OUT_SPMD}_reload_eval" \
  --init-npz "$OUT_SPMD/latest.npz" \
  --batch-samples 64 \
  --steps 0 \
  --eval-batches 16 \
  --dtype bf16 \
  --h-dim 256 \
  --num-layers 7 \
  --latent-dim "1 256" \
  --weight-joints-delta 1.0 \
  --weight-transl-delta 1.0 \
  --weight-orient-delta 2.0 \
  --weight-smpl-joints-rec 1.0 \
  --weight-joints-consistency 0.1 \
  --smpl-model-dir "$SMPL" \
  --fail-on-nonfinite

## Package outputs

In [ ]:
!cd /kaggle/working && zip -r dart_jax_vae_tpu_outputs.zip runs